# All-Prompts Evaluation Results

Head-to-head comparison of all five LLM prompt variants against the two-rater human ground truth on the 10 struggling students:

- V1 Baseline
- V1 Enriched (V1 + mental model)
- V2 Baseline
- V2 Enriched (V2 + mental model)
- V3 (CCPP)

Methodology matches `v3_prompt_eval_results.ipynb`: both-empty problems count as F1=1.0 and Jaccard=1.0; only the 18 KCs in `KC_COLUMNS` are valid. Common items are the intersection of all six rater sets (2 humans + 5 LLMs).

In [1]:
import json
import math
import os
import sys
from functools import reduce
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

ROOT = Path('/mnt/d/Projects/kintsugi')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from utils.metrics import evaluate_pair, KC_COLUMNS

HUMAN_DIR = ROOT / 'dataset' / 'Rater_KC_Tags' / 'Rated_KC_V3'
RESULTS_DIR = ROOT / 'results' / 'human_validation'
OUTPUT_DIR = RESULTS_DIR / 'all_prompts_eval_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STUDENT_IDS = ['10155', '9948', '14189', '14352', '14362', '14363', '14374', '14414', '14474', '14499']
VALID_KCS = set(KC_COLUMNS)

LLM_VARIANTS = [
    ('V1 Baseline',  'llm_v1_baseline_10students',  'llm_v1_baseline_annotations'),
    ('V1 Enriched',  'llm_v1_enriched_10students',  'llm_v1_enriched_annotations'),
    ('V2 Baseline',  'llm_v2_baseline_10students',  'llm_v2_baseline_annotations'),
    ('V2 Enriched',  'llm_v2_enriched_10students',  'llm_v2_enriched_annotations'),
    ('V3 (CCPP)',    'llm_v3_10students',           'llm_v3_annotations'),
]


def find_one(pattern, directory):
    matches = sorted(directory.glob(pattern))
    if not matches:
        raise FileNotFoundError(f'No file matched {pattern} in {directory}')
    return matches[-1]


def normalize_gaps(value):
    if isinstance(value, dict):
        gaps = value.get('gaps', [])
    elif isinstance(value, list):
        gaps = value
    else:
        gaps = []
    if not isinstance(gaps, list):
        return set()
    return {g for g in gaps if g in VALID_KCS}


def load_annotation_file(path):
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    sid = str(data.get('studentId', data.get('student_id', 'unknown')))
    anns = {f'{sid}_{pid}': normalize_gaps(v) for pid, v in data.get('annotations', {}).items()}
    return sid, anns


def merge_files(file_map):
    merged = {}
    for expected_sid, path in file_map.items():
        loaded_sid, anns = load_annotation_file(path)
        if loaded_sid != expected_sid:
            raise ValueError(f'Expected student {expected_sid}, found {loaded_sid} in {path.name}')
        merged.update(anns)
    return merged


human_a_files = {sid: find_one(f'kc_annotations_Pranay Ghuge_{sid}_*.json', HUMAN_DIR) for sid in STUDENT_IDS}
human_b_files = {sid: find_one(f'kc_annotations_Arundhati Das_{sid}_*.json', HUMAN_DIR) for sid in STUDENT_IDS}
human_a = merge_files(human_a_files)
human_b = merge_files(human_b_files)

llm_anns = {}
for label, subdir, prefix in LLM_VARIANTS:
    files = {sid: RESULTS_DIR / subdir / f'{prefix}_{sid}.json' for sid in STUDENT_IDS}
    for sid, fpath in files.items():
        if not fpath.exists():
            raise FileNotFoundError(f'{label}: missing {fpath}')
    llm_anns[label] = merge_files(files)
    print(f'Loaded {label}: {len(llm_anns[label])} student-problem annotations')

all_sets = [set(human_a), set(human_b)] + [set(v) for v in llm_anns.values()]
common_items = sorted(
    reduce(lambda a, b: a & b, all_sets),
    key=lambda x: (int(x.split('_')[0]), int(x.split('_')[1])),
)
print(f'\nCommon problem-annotations across all 6 raters: {len(common_items)}')


Loaded V1 Baseline: 372 student-problem annotations
Loaded V1 Enriched: 372 student-problem annotations
Loaded V2 Baseline: 372 student-problem annotations
Loaded V2 Enriched: 372 student-problem annotations


Loaded V3 (CCPP): 372 student-problem annotations

Common problem-annotations across all 6 raters: 372


## Per-Variant Comparison (vs Human A, Human B, Avg Human)

In [2]:
# Human ceiling (A vs B)
ceiling = evaluate_pair(human_a, human_b, common_items, KC_COLUMNS)
ceiling['Comparison'] = 'H-A vs H-B (Ceiling)'

rows = [ceiling]
for label, _, _ in LLM_VARIANTS:
    llm = llm_anns[label]
    ha = evaluate_pair(human_a, llm, common_items, KC_COLUMNS)
    hb = evaluate_pair(human_b, llm, common_items, KC_COLUMNS)
    ha['Comparison'] = f'HA vs {label}'
    hb['Comparison'] = f'HB vs {label}'
    avg = {
        'Comparison': f'AvgHuman vs {label}',
        'Cohen_kappa': (ha['Cohen_kappa'] + hb['Cohen_kappa']) / 2,
        'Gwet_AC1': (ha['Gwet_AC1'] + hb['Gwet_AC1']) / 2,
        'AC1_CI_low': None,
        'AC1_CI_high': None,
        'Problem_F1': (ha['Problem_F1'] + hb['Problem_F1']) / 2,
        'Jaccard': (ha['Jaccard'] + hb['Jaccard']) / 2,
    }
    rows.extend([ha, hb, avg])

detail_df = pd.DataFrame(rows)[['Comparison', 'Problem_F1', 'Jaccard', 'Cohen_kappa', 'Gwet_AC1']].rename(columns={
    'Problem_F1': 'F1', 'Jaccard': 'Problem Jaccard', 'Cohen_kappa': 'Kappa', 'Gwet_AC1': 'Gwet AC1',
})
detail_df.style.format({'F1': '{:.3f}', 'Problem Jaccard': '{:.3f}', 'Kappa': '{:.3f}', 'Gwet AC1': '{:.3f}'})

,Comparison,F1,Problem Jaccard,Kappa,Gwet AC1
0,H-A vs H-B (Ceiling),0.885,0.851,0.669,0.963
1,HA vs V1 Baseline,0.749,0.706,0.405,0.930
2,HB vs V1 Baseline,0.715,0.678,0.311,0.922
3,AvgHuman vs V1 Baseline,0.732,0.692,0.358,0.926
4,HA vs V1 Enriched,0.743,0.701,0.418,0.932
5,HB vs V1 Enriched,0.716,0.677,0.345,0.927
6,AvgHuman vs V1 Enriched,0.730,0.689,0.382,0.930
7,HA vs V2 Baseline,0.772,0.732,0.435,0.940
8,HB vs V2 Baseline,0.744,0.712,0.359,0.935
9,AvgHuman vs V2 Baseline,0.758,0.722,0.397,0.938


## Head-to-Head: All Prompts vs Human Ceiling

One row per LLM variant. Columns show the LLM's average performance against the two humans, the gap to the human-vs-human ceiling, and what fraction of the ceiling the LLM achieves.

In [3]:
metrics = ['Problem_F1', 'Jaccard', 'Cohen_kappa', 'Gwet_AC1']
summary_rows = []
for label, _, _ in LLM_VARIANTS:
    avg_row = next(r for r in rows if r['Comparison'] == f'AvgHuman vs {label}')
    for m in metrics:
        summary_rows.append({
            'Variant': label,
            'Metric': m,
            'Human Ceiling': ceiling[m],
            'LLM Avg': avg_row[m],
            'Gap': ceiling[m] - avg_row[m],
            '% of Ceiling': avg_row[m] / ceiling[m] if ceiling[m] > 0 else math.nan,
        })
summary_df = pd.DataFrame(summary_rows)

# Pivot for compact view: variants as rows, metrics as columns showing LLM Avg and % of ceiling
pivot_avg = summary_df.pivot(index='Variant', columns='Metric', values='LLM Avg')[metrics]
pivot_pct = summary_df.pivot(index='Variant', columns='Metric', values='% of Ceiling')[metrics]
pivot_avg.columns = [f'{m} (Avg)' for m in pivot_avg.columns]
pivot_pct.columns = [f'{m} (% Ceil)' for m in pivot_pct.columns]

# Preserve order of LLM_VARIANTS
order = [lbl for lbl, _, _ in LLM_VARIANTS]
pivot_avg = pivot_avg.reindex(order)
pivot_pct = pivot_pct.reindex(order)

headline = pd.concat([pivot_avg, pivot_pct], axis=1)
headline.loc['Human Ceiling'] = [ceiling[m] for m in metrics] + [1.0] * len(metrics)
headline = headline.loc[order + ['Human Ceiling']]

fmt_dict = {c: '{:.3f}' for c in headline.columns if '% Ceil' not in c}
fmt_dict.update({c: '{:.1%}' for c in headline.columns if '% Ceil' in c})
headline.style.format(fmt_dict)

,Problem_F1 (Avg),Jaccard (Avg),Cohen_kappa (Avg),Gwet_AC1 (Avg),Problem_F1 (% Ceil),Jaccard (% Ceil),Cohen_kappa (% Ceil),Gwet_AC1 (% Ceil)
Variant,,,,,,,,
V1 Baseline,0.732,0.692,0.358,0.926,82.7%,81.3%,53.5%,96.1%
V1 Enriched,0.730,0.689,0.382,0.930,82.5%,81.0%,57.1%,96.5%
V2 Baseline,0.758,0.722,0.397,0.938,85.7%,84.8%,59.4%,97.3%
V2 Enriched,0.749,0.714,0.397,0.937,84.7%,83.9%,59.4%,97.2%
V3 (CCPP),0.839,0.806,0.557,0.953,94.8%,94.7%,83.2%,98.9%
Human Ceiling,0.885,0.851,0.669,0.963,100.0%,100.0%,100.0%,100.0%


## Ranking by Average % of Ceiling

In [4]:
ranking = summary_df.pivot(index='Variant', columns='Metric', values='% of Ceiling').reindex(order)
ranking['Mean % Ceiling'] = ranking[metrics].mean(axis=1)
ranking = ranking.sort_values('Mean % Ceiling', ascending=False)
ranking.style.format('{:.1%}')

Metric,Cohen_kappa,Gwet_AC1,Jaccard,Problem_F1,Mean % Ceiling
Variant,,,,,
V3 (CCPP),83.2%,98.9%,94.7%,94.8%,92.9%
V2 Baseline,59.4%,97.3%,84.8%,85.7%,81.8%
V2 Enriched,59.4%,97.2%,83.9%,84.7%,81.3%
V1 Enriched,57.1%,96.5%,81.0%,82.5%,79.3%
V1 Baseline,53.5%,96.1%,81.3%,82.7%,78.4%


## Save Results

In [5]:
detail_df.to_csv(OUTPUT_DIR / 'all_prompts_detail.csv', index=False)
summary_df.to_csv(OUTPUT_DIR / 'all_prompts_summary.csv', index=False)
headline.to_csv(OUTPUT_DIR / 'all_prompts_headline.csv')
ranking.to_csv(OUTPUT_DIR / 'all_prompts_ranking.csv')

payload = {
    'n_common_items': len(common_items),
    'kc_columns': KC_COLUMNS,
    'student_ids': STUDENT_IDS,
    'human_ceiling': {k: ceiling[k] for k in ['Problem_F1', 'Jaccard', 'Cohen_kappa', 'Gwet_AC1', 'AC1_CI_low', 'AC1_CI_high']},
    'variants': {
        label: {
            'avg_vs_humans': {m: next(r for r in rows if r['Comparison'] == f'AvgHuman vs {label}')[m] for m in metrics},
            'vs_ha': {m: next(r for r in rows if r['Comparison'] == f'HA vs {label}')[m] for m in metrics},
            'vs_hb': {m: next(r for r in rows if r['Comparison'] == f'HB vs {label}')[m] for m in metrics},
        }
        for label, _, _ in LLM_VARIANTS
    },
    'methodology': 'both-empty = F1=1.0 and Jaccard=1.0; common items = intersection across all 6 raters',
}
with (OUTPUT_DIR / 'all_prompts_eval_summary.json').open('w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2)

print(f'Saved to {OUTPUT_DIR}')

Saved to /mnt/d/Projects/kintsugi/results/human_validation/all_prompts_eval_results
